# Three-Arm Benchmark: SOTA Configuration vs Baseline

**Author**: Claude (executor)<br>
**Date**: 2026-07-07<br>
**Purpose**: Measure the promoted SOTA retrieval configuration against the recorded baseline on a single comparable metric - pure-seed evidence recall over the 24 gold CPAP probes (H34/H53 convention). Three arms isolate the pure retrieval lever from the full-stack rebuild.

## Arms
- **Arm 1 - baseline (recorded)**: the baseline graph on `neo4j2`, retrieval at production `top_k=8`. Its recorded quality is v28 = **0.90 hybrid, 60/63 deterministic** (v1 multidoc harness, `@archive/kgf-v1` - a *different* metric, cited only as provenance of the baseline graph). Here we also measure its pure-seed recall@8 as the comparable baseline.
- **Arm 2 - pure lever**: the SAME baseline graph, retrieval at `top_k=16` (the only executed-and-promoted retrieval lever, H53). `Arm2 - Arm1` is the pure retrieval-lever delta - the honest headline for "SOTA config vs baseline".
- **Arm 3 - SOTA rebuild**: the freshly rebuilt graph on the default `neo4j` instance, built with `config-sota.yml` (local gpt-oss-120b extractor, `top_k=16`). `Arm3 - Arm1` is the full-stack delta.

> **Known confound (Arm 3)**: the extractor engine differs from baseline - the local gpt-oss-120b model cured ~49 types vs baseline ~24. So `Arm3 - Arm1` mixes the retrieval lever with a full extraction-engine swap, while `Arm2 - Arm1` isolates the lever cleanly.

All Neo4j access is READ-ONLY (MATCH ... RETURN + vector index queries). One Bedrock Titan embedding per probe for the query vector; no LLM judge is used - the metric is deterministic string matching.

In [1]:
# Imports
# stdlib
import json  # report persistence
import re  # tokenization + gold matching
import datetime  # UTC report stamp
from copy import deepcopy  # per-arm settings override
from pathlib import Path  # paths

# third party
import yaml  # probe set
from rich import print as rprint  # semantic output
from rich.table import Table  # comparison table
from rich.console import Console  # table render

# project
from knowledge_graph_foundry import load_settings  # settings loader
from knowledge_graph_foundry.extraction import generate_embeddings  # Titan query embedding
from knowledge_graph_foundry.graph.graphrag import vector_query  # dense retrieval channel
from knowledge_graph_foundry.models import Entity  # embedding probe wrapper
from knowledge_graph_foundry.pipeline import Foundry  # read-only driver context

console = Console()

2026-07-07 14:34:38.665 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

Probe set, base settings, and the three arm definitions. The vector index and embedding provider are shared across arms; only the graph URI and `top_k` change.

In [2]:
# Configuration
PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")  # gold-evidence CPAP probes
base = load_settings(Path("../config.yml"))               # base settings (Bedrock Titan embeddings)
VEC_INDEX = base.graphrag.vector_index_name               # kgf_entity_embeddings

probes = [p for p in yaml.safe_load(PROBES_PATH.read_text()) if p.get("gold_evidence")]

BASELINE_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"  # Arm 1 / Arm 2 graph
SOTA_URI = "bolt://user-konrad.jelen-kgf-neo4j:7687"       # Arm 3 graph

# recorded v28 baseline (v1 multidoc harness - different metric, provenance only)
RECORDED_V28 = {"hybrid": 0.90, "deterministic": "60/63"}

rprint(f"""[bold cyan]Configuration[/bold cyan]
[bold]Probes[/bold]  {PROBES_PATH.name}  ([yellow]{len(probes)}[/yellow] with gold evidence)
[bold]Vector index[/bold]  {VEC_INDEX}   [bold]Embeddings[/bold]  {base.embeddings.provider}/{base.embeddings.model}
[bold]Arm 1[/bold]  baseline {BASELINE_URI.split('-')[-1]}  top_k=8   [dim](recorded v28: 0.90 hybrid, 60/63 det)[/dim]
[bold]Arm 2[/bold]  baseline {BASELINE_URI.split('-')[-1]}  top_k=16  [dim](H53 pure lever)[/dim]
[bold]Arm 3[/bold]  SOTA {SOTA_URI.split('-')[-1]}  top_k=16  [dim](config-sota.yml, local extractor - confounded)[/dim]
[bold]Metric[/bold]  mean fraction of gold_evidence strings present in top_k vector-seed renders (deterministic)""")

Configuration
Probes  cpap-probe-set.yml  (24 with gold evidence)
Vector index  kgf_entity_embeddings   Embeddings  bedrock/amazon.titan-embed-text-v2:0
Arm 1  baseline neo4j2:7687  top_k=8   (recorded v28: 0.90 hybrid, 60/63 det)
Arm 2  baseline neo4j2:7687  top_k=16  (H53 pure lever)
Arm 3  SOTA neo4j:7687  top_k=16  (config-sota.yml, local extractor - confounded)
Metric  mean fraction of gold_evidence strings present in top_k vector-seed renders (deterministic)

## Harness

Pure-seed evidence-recall functions copied verbatim from the H53 / H34 convention: the gold matcher (`present`), the pure-seed node render (`render_nodes`, alias merge off, SIMILAR_TO excluded, currently-valid relations only), and the per-arm runner. `run_arm` mutates a copy of the base settings to point at one graph at one `top_k`, then measures recall over every probe.

In [3]:
# Gold matcher - H34 iteration-3 conventions
def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())

_UNIT = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"

def present(gold, ctx_norm):
    ng = _norm(gold)
    if ng in ctx_norm:
        return True
    squashed = re.sub(r"[\s,()]", "", ctx_norm)
    skeleton = re.sub(r"[\s,()]", "", re.sub(_UNIT, "", ng))
    if any(c.isdigit() for c in skeleton) and len(skeleton) >= 5 and skeleton in squashed:
        return True
    toks = re.findall(r"[\w.\-/]*\d[\w.\-/]*", gold)
    if toks:
        hit = sum(1 for t in toks if _norm(t) in ctx_norm or re.sub(r"[\s,()]", "", _norm(t)) in squashed)
        return hit >= max(1, len(toks) // 2 + (len(toks) % 2))
    words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng))
    cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx_norm))
    return bool(words) and len(words & cw) / len(words) >= 0.6


def render_nodes(session, node_ids):
    """H34 pure-seed render: description + prop_* spec + currently-valid relations, no alias merge."""
    blocks = []
    for nid in node_ids:
        row = session.run(
            "MATCH (e:Entity {id:$id}) RETURN e.name AS name, labels(e) AS types, "
            "e.description AS description, properties(e) AS props", id=nid).single()
        if row is None:
            continue
        spec = {k.removeprefix("prop_"): v for k, v in row["props"].items() if k.startswith("prop_")}
        rels = session.run(
            "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL "
            "AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15", id=nid).data()
        blocks.append(
            f"## {row['name']} ({', '.join(row['types'])})\n{row['description'] or ''}\n"
            f"Properties: {json.dumps(spec, default=str)}\n"
            "Relations: " + "; ".join(f"{r['rel']} -> {r['name']}" for r in rels))
    return _norm("\n".join(blocks))


def fingerprint(driver):
    with driver.session() as s:
        ents = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
        rels = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
        docs = s.run("MATCH (d:Document) RETURN count(d) AS c").single()["c"]
    return {"entities": ents, "relationships": rels, "documents": docs}


def run_arm(uri, top_k, label):
    st = deepcopy(base)
    st.neo4j.uri, st.neo4j.user, st.neo4j.password = uri, "neo4j", "kgfoundry"
    st.graphrag.top_k = top_k
    per = {}
    with Foundry(st) as f:
        fp = fingerprint(f.driver)  # READ-ONLY
        for p in probes:
            q = p["question"]
            pe = Entity.create(q[:80], types=["Query"], description=q)
            emb = generate_embeddings([pe], st.embeddings)[0].embedding
            seeds = [s["id"] for s in vector_query(f.driver, emb, VEC_INDEX, top_k=top_k)]
            with f.driver.session() as sess:
                ctx = render_nodes(sess, seeds)
            golds = p["gold_evidence"]
            per[p["id"]] = {"recall": sum(present(g, ctx) for g in golds) / len(golds), "n_gold": len(golds)}
    n = len(per)
    mean = sum(v["recall"] for v in per.values()) / n
    full = sum(1 for v in per.values() if v["recall"] == 1.0)
    return {"label": label, "uri": uri, "top_k": top_k, "fingerprint": fp,
            "mean_recall": mean, "fully_covered": full, "n_probes": n, "per_probe": per}

## Arm 1 - baseline @ top_k=8

The baseline graph on `neo4j2` at the production seed budget. This is the comparable retrieval baseline; the recorded v28 quality (0.90 hybrid, 60/63 deterministic) is cited as the graph's provenance from the v1 harness.

In [4]:
arm1 = run_arm(BASELINE_URI, 8, "Arm1 baseline@k8")
rprint(f"[bold]Arm 1[/bold]  graph {arm1['fingerprint']}")
rprint(f"  mean evidence recall [yellow]{arm1['mean_recall']:.4f}[/yellow]   fully-covered [yellow]{arm1['fully_covered']}/{arm1['n_probes']}[/yellow]")

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Document` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (d:Document) RETURN count(d) AS c'


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


2026-07-07 14:34:46.097 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-07 14:34:46.099 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)


Arm 1  graph {'entities': 2798, 'relationships': 31593, 'documents': 0}

mean evidence recall 0.6667   fully-covered 13/24

## Arm 2 - pure lever @ top_k=16

The SAME baseline graph, seed budget doubled to 16. Nothing else changes, so the delta versus Arm 1 is attributable entirely to the H53 retrieval lever.

In [5]:
arm2 = run_arm(BASELINE_URI, 16, "Arm2 lever@k16")
rprint(f"[bold]Arm 2[/bold]  graph {arm2['fingerprint']}")
rprint(f"  mean evidence recall [yellow]{arm2['mean_recall']:.4f}[/yellow]   fully-covered [yellow]{arm2['fully_covered']}/{arm2['n_probes']}[/yellow]")

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Document` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (d:Document) RETURN count(d) AS c'


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


2026-07-07 14:34:46.159 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


2026-07-07 14:34:47.959 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.016 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


Arm 2  graph {'entities': 2798, 'relationships': 31593, 'documents': 0}

mean evidence recall 0.8542   fully-covered 20/24

## Arm 3 - SOTA rebuild @ top_k=16 (and k=8 reference)

The freshly rebuilt graph on the default `neo4j` instance, built with `config-sota.yml`. Arm 3b (`top_k=8` on the same graph) isolates how much of any Arm 3 movement is the lever versus the extractor swap. **Confound**: the local gpt-oss-120b extractor produces a different entity/type surface than the baseline's engine.

In [6]:
arm3 = run_arm(SOTA_URI, 16, "Arm3 SOTA@k16")
arm3b = run_arm(SOTA_URI, 8, "Arm3b SOTA@k8 (ref)")
rprint(f"[bold]Arm 3[/bold]   graph {arm3['fingerprint']}")
rprint(f"  @k16 mean recall [yellow]{arm3['mean_recall']:.4f}[/yellow]  fully-covered [yellow]{arm3['fully_covered']}/{arm3['n_probes']}[/yellow]")
rprint(f"  @k8  mean recall [yellow]{arm3b['mean_recall']:.4f}[/yellow]  fully-covered [yellow]{arm3b['fully_covered']}/{arm3b['n_probes']}[/yellow]  [dim](lever on SOTA graph: {arm3['mean_recall']-arm3b['mean_recall']:+.4f})[/dim]")

2026-07-07 14:34:48.093 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.163 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


2026-07-07 14:34:48.222 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.271 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


2026-07-07 14:34:48.318 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.365 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


2026-07-07 14:34:48.429 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.476 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


2026-07-07 14:34:48.538 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:48.585 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


...

2026-07-07 14:34:49.881 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:49.908 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


2026-07-07 14:34:49.934 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:49.961 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:49.986 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:50.011 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:50.033 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-07 14:34:50.056 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache


Arm 3   graph {'entities': 3693, 'relationships': 24570, 'documents': 76}

@k16 mean recall 0.7292  fully-covered 14/24

@k8  mean recall 0.6875  fully-covered 14/24  (lever on SOTA graph: +0.0417)

## Comparison and deltas

Single-metric comparison across arms, the two headline deltas, per-probe flips, and the persisted JSON report.

In [7]:
def flips(a, b):
    out = {}
    for pid in a["per_probe"]:
        ra = a["per_probe"][pid]["recall"]
        rb = b["per_probe"].get(pid, {}).get("recall", 0.0)
        if abs(rb - ra) >= 1e-9:
            out[pid] = round(rb - ra, 4)
    return out

d21 = arm2["mean_recall"] - arm1["mean_recall"]
d31 = arm3["mean_recall"] - arm1["mean_recall"]

tbl = Table(title="Three-Arm Benchmark - pure-seed evidence recall (24 gold probes)")
for c in ["Arm", "Graph", "top_k", "entities", "rels", "mean recall", "full/24", "vs Arm1"]:
    tbl.add_column(c)
tbl.add_row("1 baseline", "neo4j2", "8", str(arm1["fingerprint"]["entities"]), str(arm1["fingerprint"]["relationships"]), f"{arm1['mean_recall']:.4f}", f"{arm1['fully_covered']}", "-")
tbl.add_row("2 lever", "neo4j2", "16", str(arm2["fingerprint"]["entities"]), str(arm2["fingerprint"]["relationships"]), f"{arm2['mean_recall']:.4f}", f"{arm2['fully_covered']}", f"{d21:+.4f} ({d21/arm1['mean_recall']:+.1%})")
tbl.add_row("3 SOTA", "neo4j", "16", str(arm3["fingerprint"]["entities"]), str(arm3["fingerprint"]["relationships"]), f"{arm3['mean_recall']:.4f}", f"{arm3['fully_covered']}", f"{d31:+.4f} ({d31/arm1['mean_recall']:+.1%})")
tbl.add_row("3b SOTA ref", "neo4j", "8", str(arm3b["fingerprint"]["entities"]), str(arm3b["fingerprint"]["relationships"]), f"{arm3b['mean_recall']:.4f}", f"{arm3b['fully_covered']}", "-")
console.print(tbl)

rprint(f"\n[bold]Recorded v28 baseline (provenance, v1 harness):[/bold] {RECORDED_V28['hybrid']:.2f} hybrid, {RECORDED_V28['deterministic']} deterministic")
rprint(f"[bold green]HEADLINE (honest, Arm2-Arm1):[/bold green] pure lever {d21:+.4f} ({d21/arm1['mean_recall']:+.1%} relative)")
rprint(f"[bold]Full stack (Arm3-Arm1):[/bold] {d31:+.4f} ({d31/arm1['mean_recall']:+.1%} relative)")
rprint(f"[dim]Arm2 vs Arm1 flips:[/dim] {flips(arm1, arm2)}")
rprint(f"[dim]Arm3 vs Arm1 flips:[/dim] {flips(arm1, arm3)}")

               Three-Arm Benchmark - pure-seed evidence recall (24 gold probes)               
┏━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Arm         ┃ Graph  ┃ top_k ┃ entities ┃ rels  ┃ mean recall ┃ full/24 ┃ vs Arm1          ┃
┡━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 1 baseline  │ neo4j2 │ 8     │ 2798     │ 31593 │ 0.6667      │ 13      │ -                │
│ 2 lever     │ neo4j2 │ 16    │ 2798     │ 31593 │ 0.8542      │ 20      │ +0.1875 (+28.1%) │
│ 3 SOTA      │ neo4j  │ 16    │ 3693     │ 24570 │ 0.7292      │ 14      │ +0.0625 (+9.4%)  │
│ 3b SOTA ref │ neo4j  │ 8     │ 3693     │ 24570 │ 0.6875      │ 14      │ -                │
└─────────────┴────────┴───────┴──────────┴───────┴─────────────┴─────────┴──────────────────┘

Recorded v28 baseline (provenance, v1 harness): 0.90 hybrid, 60/63 deterministic

HEADLINE (honest, Arm2-Arm1): pure lever +0.1875 (+28.1% relative)

Full stack (Arm3-Arm1): +0.0625 (+9.4% relative)

Arm2 vs Arm1 flips: {'P01': 1.0, 'P03': 1.0, 'P11': 0.5, 'P12': 0.5, 'P15': 0.5, 'P17': 0.5, 'P18': 0.5}

Arm3 vs Arm1 flips: {'P01': 0.5, 'P03': 1.0, 'P08': -1.0, 'P12': 0.5, 'P14': -0.5, 'P19': 1.0}

In [8]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "generated_utc": stamp,
    "harness": "pure-seed evidence recall (H34/H53 convention), 24 gold CPAP probes",
    "metric": "mean fraction of gold_evidence strings present in top_k vector-seed renders (deterministic, no LLM judge)",
    "recorded_baseline_v28": {**RECORDED_V28, "note": "v1 multidoc benchmark harness (@archive/kgf-v1); different metric, cited as provenance of the Arm 1 baseline graph, not re-run"},
    "arms": {"arm1_baseline_k8": arm1, "arm2_lever_k16": arm2, "arm3_sota_k16": arm3, "arm3b_sota_k8_ref": arm3b},
    "deltas": {
        "arm2_minus_arm1_mean_recall": round(d21, 4),
        "arm2_minus_arm1_relative": round(d21 / arm1["mean_recall"], 4),
        "arm3_minus_arm1_mean_recall": round(d31, 4),
        "arm3_minus_arm1_relative": round(d31 / arm1["mean_recall"], 4),
        "lever_on_sota_graph": round(arm3["mean_recall"] - arm3b["mean_recall"], 4),
    },
    "per_probe_flips": {"arm2_vs_arm1": flips(arm1, arm2), "arm3_vs_arm1": flips(arm1, arm3)},
}
outp = Path(f"../reports/three-arm-benchmark-{stamp}.json")
outp.write_text(json.dumps(report, indent=2))
rprint(f"[bold]Wrote[/bold] {outp}")

Wrote ../reports/three-arm-benchmark-20260707T123450Z.json